"""
========================================================================
NOTEBOOK 04: PAPER FIGURES
========================================================================

Three figures for the workshop submission:
  Figure 1 (main):    Cross-model affective deviation, by desk × prompt
  Figure 2 (scatter): Per-headline NYT vs LLM sentiment, paired comparison
  Figure 3 (lexical): Top overused/underused words, GPT vs NYT heatmap

All figures use self-contained CSV loads — no kernel state dependency.
Save as PDF (vector, scales for paper) and PNG (preview).

Outputs go to:
  data/results/figures/
    fig1_crossmodel_deviation.pdf
    fig2_per_headline_scatter.pdf
    fig3_lexical_fingerprint.pdf
"""


In [ ]:

# =============================================================================
# CELL 1: Setup
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10, "axes.labelsize": 10, "axes.titlesize": 11,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
})

ROOT = Path("..").resolve()
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "data" / "results"
FIGS = RESULTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

# Model display labels + colors (single source of truth)
M_GPT, M_G25, M_G3, M_CLAUDE = "GPT-4o-mini", "Gemini 2.5 Flash", "Gemini 3 Flash", "Claude Haiku 4.5"
color_map = {M_GPT: "#C44E52", M_G25: "#4C72B0", M_G3: "#55A868", M_CLAUDE: "#8172B3"}
# map display label -> the model string used in the results CSVs
DATA_NAME = {M_GPT: "GPT-4o-mini", M_G25: "Gemini-2.5-Flash", M_CLAUDE: "Claude-Haiku-4.5"}

# Load data
nyt_axis2 = pd.read_csv(PROCESSED / "nyt_baseline_axis2.csv")
gpt_axis2 = pd.read_csv(RESULTS / "axis2_gpt.csv")
gemini_axis2 = pd.read_csv(RESULTS / "axis2_gemini.csv")          # Gemini-2.5-Flash (n=40)
sampled = pd.read_csv(PROCESSED / "experiment_sample.csv")

# Optional extra models (load if present)
g3_path = RESULTS / "axis2_gemini_3flash.csv"
gemini3_axis2 = pd.read_csv(g3_path) if g3_path.exists() else None    # gemini-3-flash-preview, n=10
c_path = RESULTS / "axis2_claude.csv"
claude_axis2 = pd.read_csv(c_path) if c_path.exists() else None      # Claude-Haiku-4.5, n=40

# Shared helpers
nyt_agg = nyt_axis2[["sample_id", "news_desk", "sentiment"]].rename(
    columns={"sentiment": "nyt_sentiment"})

def aggregate(df, model_label):
    agg = df.groupby(["sample_id", "prompt_version", "news_desk"],
                     as_index=False)["sentiment"].mean()
    agg["model"] = model_label
    return agg

def add_deviation(llm_agg):
    return llm_agg.merge(nyt_agg[["sample_id", "nyt_sentiment"]], on="sample_id").assign(
        deviation=lambda d: d["sentiment"] - d["nyt_sentiment"])

def forest_plot(summary, model_order, title, fname, xlim=(-0.5, 1.6)):
    ordered = []
    for model in model_order:
        for prompt in ["A", "E"]:
            for desk in ["Foreign", "Business"]:
                row = summary[(summary.model == model) & (summary.prompt_version == prompt)
                              & (summary.news_desk == desk)]
                if len(row): ordered.append(row.iloc[0])
    fdf = pd.DataFrame(ordered).reset_index(drop=True)
    fdf["label"] = fdf.apply(lambda r: f"{r['model']}, V.{r['prompt_version']}, {r['news_desk']}", axis=1)
    fig, ax = plt.subplots(figsize=(7, 0.42 * len(fdf) + 1.4))
    yps = np.arange(len(fdf))[::-1]
    for i, (_, row) in enumerate(fdf.iterrows()):
        color = color_map[row["model"]]
        marker = "o" if row["prompt_version"] == "A" else "s"
        fc = color if row["prompt_version"] == "A" else "white"
        ax.errorbar(row["mean_dev"], yps[i], xerr=row["se_dev"], fmt=marker, markersize=9,
                    color=color, markerfacecolor=fc, markeredgecolor=color,
                    markeredgewidth=2, capsize=4, elinewidth=1.5)
    ax.axvline(0, color="gray", linestyle="--", linewidth=1, alpha=0.7, zorder=0)
    ax.set_yticks(yps); ax.set_yticklabels(fdf["label"])
    ax.set_xlabel("Mean per-headline deviation (LLM \u2212 NYT)", fontsize=11)
    ax.set_title(title, fontsize=10, pad=10)
    leg = [Line2D([0],[0],marker="o",color="w",markerfacecolor=color_map[m],
                  markeredgecolor=color_map[m],markersize=10,label=m) for m in model_order]
    leg += [Line2D([0],[0],marker="o",color="w",markerfacecolor="black",markeredgecolor="black",
                   markersize=10,label="V.A (default)"),
            Line2D([0],[0],marker="s",color="w",markerfacecolor="white",markeredgecolor="black",
                   markersize=10,markeredgewidth=2,label="V.E (intervention)")]
    ax.legend(handles=leg, loc="lower left", frameon=False, fontsize=8.5, ncol=1, handletextpad=0.4)
    ax.text(0.02, len(fdf) - 0.5, "NYT", color="gray", fontsize=9, fontstyle="italic")
    ax.set_xlim(*xlim); plt.tight_layout()
    fig.savefig(FIGS / f"{fname}.pdf", bbox_inches="tight")
    fig.savefig(FIGS / f"{fname}.png", bbox_inches="tight", dpi=200)
    plt.show()
    return fdf

# Per-headline aggregates (reused by several figures)
gpt_agg = aggregate(gpt_axis2, M_GPT)
gem_agg = aggregate(gemini_axis2, M_G25)
claude_agg = aggregate(claude_axis2, M_CLAUDE) if claude_axis2 is not None else None

print("Data loaded:")
print(f"  NYT:            {len(nyt_axis2)} rows")
print(f"  GPT-4o-mini:    {len(gpt_axis2)} rows")
print(f"  Gemini-2.5:     {len(gemini_axis2)} rows")
print(f"  Gemini-3-flash: {len(gemini3_axis2) if gemini3_axis2 is not None else 'N/A'} rows")
print(f"  Claude-Haiku:   {len(claude_axis2) if claude_axis2 is not None else 'N/A'} rows")

In [ ]:

# =============================================================================
# CELL 0b: Figure 0 - Three-layer framework (conceptual anchor)
# Maps the empirical results onto the reproduction / recovery / miscalibration
# triad. Numbers come from Notebooks 02-04 (n=40/desk, 3 model families).
# =============================================================================

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(8.4, 5.4))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

layers = [
    dict(y=6.8, name="Reproduced", tag="reproduced by default",
         comp="Entity & topic vocabulary",
         ev="All 3 models hit desk-specific lexicons spontaneously\n"
            "(Business fidelity \u2248 NYT; Foreign fidelity positive).",
         fill="#DCEFE0", edge="#3C8C57", icon="\u2713"),
    dict(y=3.7, name="Recoverable", tag="recovered by prompting",
         comp="Business-desk affective baseline",
         ev="Prompt scaffolding (V.E) recovers the Business tone \u2014\n"
            "Claude reaches NYT (n.s., p = 0.23).",
         fill="#FBEFD2", edge="#C8922A", icon="\u2248"),
    dict(y=0.6, name="Miscalibrated", tag="survives prompting",
         comp="Foreign-desk affective baseline",
         ev="Stays more positive than NYT across all 3 families, even under\n"
            "the strongest prompt.  Foreign > Business in 12/12 cells.",
         fill="#F6D9DB", edge="#B23A3F", icon="\u2717"),
]
bx, bw, bh = 1.55, 8.05, 2.5
for L in layers:
    ax.add_patch(FancyBboxPatch((bx, L["y"]), bw, bh,
                 boxstyle="round,pad=0.02,rounding_size=0.15",
                 facecolor=L["fill"], edgecolor=L["edge"], linewidth=2.2))
    ax.text(bx + 0.4, L["y"] + bh - 0.55, L["icon"], fontsize=22, color=L["edge"],
            va="center", ha="center", fontweight="bold", fontfamily="DejaVu Sans")
    ax.text(bx + 0.95, L["y"] + bh - 0.55, L["name"], fontsize=16, color=L["edge"],
            va="center", ha="left", fontweight="bold")
    ax.text(bx + bw - 0.3, L["y"] + bh - 0.55, L["tag"], fontsize=9.5, color=L["edge"],
            va="center", ha="right", fontstyle="italic")
    ax.text(bx + 0.95, L["y"] + bh - 1.2, L["comp"], fontsize=11.5, color="#1a1a1a",
            va="center", ha="left", fontweight="bold")
    ax.text(bx + 0.95, L["y"] + 0.6, L["ev"], fontsize=9.3, color="#333",
            va="center", ha="left", linespacing=1.35)

ax.add_patch(FancyArrowPatch((0.75, 9.1), (0.75, 0.7), arrowstyle="-|>",
             mutation_scale=18, color="#555", lw=2))
ax.text(0.42, 4.9, "Increasing resistance to prompt engineering",
        rotation=90, va="center", ha="center", fontsize=9.5, color="#555")

ax.set_title(
    "A three-layer diagnostic for institutional voice in LLM-generated news\n"
    "(NYT China coverage; GPT-4o-mini, Gemini-2.5-Flash, Claude-Haiku-4.5; n = 40/desk)",
    fontsize=11, pad=12)
plt.tight_layout()
fig.savefig(FIGS / "fig0_framework.pdf", bbox_inches="tight")
fig.savefig(FIGS / "fig0_framework.png", bbox_inches="tight", dpi=200)
plt.show()
print(f"Saved Figure 0 (framework) to {FIGS}")

In [ ]:

# =============================================================================
# CELL 2: Figure 1 - Cross-model deviation (MAIN forest), n=40
# GPT-4o-mini + Gemini-2.5-Flash + Claude-Haiku-4.5 (if present), prompts A/E.
# =============================================================================

parts = [add_deviation(gpt_agg), add_deviation(gem_agg)]
model_order = [M_GPT, M_G25]
if claude_agg is not None:
    parts.append(add_deviation(claude_agg))
    model_order.append(M_CLAUDE)

all_dev = pd.concat(parts, ignore_index=True)
all_dev_AE = all_dev[all_dev["prompt_version"].isin(["A", "E"])].copy()
summary = all_dev_AE.groupby(["model", "prompt_version", "news_desk"], as_index=False).agg(
    mean_dev=("deviation", "mean"), sd_dev=("deviation", "std"), n=("deviation", "count"))
summary["se_dev"] = summary["sd_dev"] / np.sqrt(summary["n"])

n_models = len(model_order)
forest_plot(
    summary, model_order,
    "Cross-model affective deviation from NYT, by prompt and desk\n"
    "(positive = more positive than NYT; bars = \u00b11 SE; n = 40 headlines/cell)",
    "fig1_crossmodel_deviation",
)
print(f"Saved Figure 1 ({n_models} models) to {FIGS}")

In [ ]:

# =============================================================================
# CELL 2b: Figure 1b — Gemini model sensitivity (same 20 headlines, n=10)
# gemini-3-flash-preview vs gemini-2.5-flash on the common sample (sample_id 0-19).
# Shows that calibration magnitude is model-version-specific.
# =============================================================================

if gemini3_axis2 is not None:
    common_ids = sorted(gemini3_axis2["sample_id"].unique())  # original 20 headlines
    g25_agg = aggregate(gemini_axis2, M_G25)
    g3_agg = aggregate(gemini3_axis2, M_G3)
    ms = pd.concat([add_deviation(g25_agg), add_deviation(g3_agg)], ignore_index=True)
    ms = ms[ms["sample_id"].isin(common_ids) & ms["prompt_version"].isin(["A", "E"])].copy()
    ms_sum = ms.groupby(["model", "prompt_version", "news_desk"], as_index=False).agg(
        mean_dev=("deviation", "mean"), sd_dev=("deviation", "std"), n=("deviation", "count"))
    ms_sum["se_dev"] = ms_sum["sd_dev"] / np.sqrt(ms_sum["n"])
    forest_plot(
        ms_sum, [M_G3, M_G25],
        "Gemini model sensitivity: same 20 headlines, two model versions\n"
        "(positive = more positive than NYT; bars = ±1 SE; n = 10 headlines/cell)",
        "fig1b_gemini_model_sensitivity",
    )
    print(f"Saved Figure 1b to {FIGS}")
else:
    print("axis2_gemini_3flash.csv not found - skipping model-sensitivity figure.")

In [ ]:

# =============================================================================
# CELL 2c: Figure 1c - Desk-asymmetry dumbbell (consolidated cross-model view)
# Each (model, prompt) row draws a line from its Business deviation to its Foreign
# deviation. Every line points right => Foreign is always more positive than
# Business (the Foreign-desk bottleneck), across all model families.
# Open marker = NOT significantly different from NYT (i.e. recovered).
# =============================================================================

pv_path = RESULTS / "results_crossmodel3_paired.csv"
if pv_path.exists():
    pv = pd.read_csv(pv_path)  # columns: model, prompt, desk, deviation, p_value
    data_order = [d for d in ["GPT-4o-mini", "Gemini-2.5-Flash", "Claude-Haiku-4.5"]
                  if d in pv["model"].unique()]
    inv_disp = {v: k for k, v in DATA_NAME.items()}

    rows = []
    for m in data_order:
        for v in ["A", "E"]:
            b = pv[(pv.model == m) & (pv.prompt == v) & (pv.desk == "Business")]
            f = pv[(pv.model == m) & (pv.prompt == v) & (pv.desk == "Foreign")]
            if len(b) and len(f):
                rows.append({"model": m, "prompt": v,
                             "biz": b.deviation.iloc[0], "for": f.deviation.iloc[0],
                             "biz_ns": bool(b.p_value.iloc[0] >= 0.05),
                             "for_ns": bool(f.p_value.iloc[0] >= 0.05)})
    D = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(7.6, 0.5 * len(D) + 1.6))
    yps = np.arange(len(D))[::-1]
    for i, (_, r) in enumerate(D.iterrows()):
        y = yps[i]
        color = color_map[inv_disp[r["model"]]]
        ax.plot([r["biz"], r["for"]], [y, y], color=color, lw=2.4, alpha=0.9, zorder=1)
        ax.scatter(r["biz"], y, marker="o", s=95, zorder=2,
                   color="white" if r["biz_ns"] else color, edgecolor=color, linewidth=2)
        ax.scatter(r["for"], y, marker=">", s=115, zorder=2,
                   color="white" if r["for_ns"] else color, edgecolor=color, linewidth=2)

    ax.axvline(0, color="gray", ls="--", lw=1, alpha=0.7, zorder=0)
    ax.text(-0.03, (len(D) - 1) / 2.0, "NYT", color="gray", fontsize=9, fontstyle="italic", ha="right", va="center", rotation=90)
    ax.set_yticks(yps)
    ax.set_yticklabels([f"{inv_disp[r['model']]}, V.{r['prompt']}" for _, r in D.iterrows()])
    ax.set_xlabel("Mean per-headline deviation from NYT (LLM \u2212 NYT)", fontsize=11)
    ax.set_title(
        "Foreign always exceeds Business, across model families\n"
        "(\u25cf Business   \u25ba Foreign;  open marker = not significant vs NYT;  n = 40/desk)",
        fontsize=10, pad=10)
    leg = [
        Line2D([0],[0], marker="o", color="w", markerfacecolor="gray", markeredgecolor="gray",
               markersize=10, label="Business desk"),
        Line2D([0],[0], marker=">", color="w", markerfacecolor="gray", markeredgecolor="gray",
               markersize=11, label="Foreign desk"),
        Line2D([0],[0], marker="o", color="w", markerfacecolor="white", markeredgecolor="gray",
               markersize=10, markeredgewidth=2, label="n.s. vs NYT (recovered)"),
    ]
    ax.legend(handles=leg, loc="lower right", frameon=False, fontsize=8.5)
    ax.set_xlim(-0.25, 1.2)
    plt.tight_layout()
    fig.savefig(FIGS / "fig1c_desk_asymmetry.pdf", bbox_inches="tight")
    fig.savefig(FIGS / "fig1c_desk_asymmetry.png", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved Figure 1c to {FIGS}")
else:
    print("results_crossmodel3_paired.csv not found - run Notebook 04 (Claude) first.")

In [ ]:


# =============================================================================
# CELL 3: Figure 2 - Per-headline scatter (paired comparison), n=40
# x: NYT sentiment per headline; y: LLM sentiment. Diagonal y=x = perfect imitation.
# =============================================================================

def to_paired(llm_agg, model_label):
    paired = llm_agg.merge(nyt_agg[["sample_id", "nyt_sentiment"]], on="sample_id")
    paired["model"] = model_label
    return paired[["sample_id", "model", "prompt_version", "news_desk",
                   "nyt_sentiment", "sentiment"]]

panel_aggs = [(M_GPT, gpt_agg), (M_G25, gem_agg)]
if claude_agg is not None:
    panel_aggs.append((M_CLAUDE, claude_agg))

paired_all = pd.concat([
    to_paired(a[a["prompt_version"].isin(["A", "E"])], m) for m, a in panel_aggs
], ignore_index=True)

models_present = [m for m, _ in panel_aggs]
fig, axes = plt.subplots(1, len(models_present), figsize=(5.4 * len(models_present), 5),
                         sharex=True, sharey=True)
if len(models_present) == 1:
    axes = [axes]
style_map = {"A": ("o", "filled"), "E": ("s", "hollow")}

for ax, model in zip(axes, models_present):
    color = color_map[model]
    sub = paired_all[paired_all["model"] == model]
    for prompt, group in sub.groupby("prompt_version"):
        marker, fill = style_map[prompt]
        biz = group[group["news_desk"] == "Business"]
        foreign = group[group["news_desk"] == "Foreign"]
        ax.scatter(biz["nyt_sentiment"], biz["sentiment"], marker=marker, s=70,
                   color=color if fill == "filled" else "white", edgecolor=color,
                   linewidth=1.8, label=f"V.{prompt}, Business", alpha=0.85)
        ax.scatter(foreign["nyt_sentiment"], foreign["sentiment"],
                   marker=marker if marker == "o" else "^", s=70,
                   color="gray" if fill == "filled" else "white",
                   edgecolor="gray" if fill == "filled" else color,
                   linewidth=1.8, label=f"V.{prompt}, Foreign", alpha=0.7)
    ax.plot([-1, 1], [-1, 1], "k--", alpha=0.3, linewidth=1, zorder=0)
    ax.text(-0.95, 0.95, "y = x", fontsize=8, color="gray", style="italic")
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("NYT real-article sentiment")
    if ax is axes[0]:
        ax.set_ylabel("LLM-generated article sentiment")
    ax.set_title(model)
    ax.axhline(0, color="lightgray", linewidth=0.5, zorder=0)
    ax.axvline(0, color="lightgray", linewidth=0.5, zorder=0)
    ax.legend(loc="lower right", fontsize=7.5, frameon=False)

fig.suptitle(
    "Per-headline LLM vs NYT sentiment (each point = one headline)\n"
    "Points above y=x: LLM is more positive than NYT for that headline",
    fontsize=11, y=1.02)
plt.tight_layout()
fig.savefig(FIGS / "fig2_per_headline_scatter.pdf", bbox_inches="tight")
fig.savefig(FIGS / "fig2_per_headline_scatter.png", bbox_inches="tight", dpi=200)
plt.show()
print(f"Saved Figure 2 ({len(models_present)} models) to {FIGS}")

In [ ]:
"""
CELL 4 (REVISED): Figure 3 — Lexical fingerprint, single-direction overused only.

After filtering noise words, the underused dimension has no robust institutional
voice signal — it's dominated by NYT brand infrastructure and bio metadata.
The real finding lives in the overused dimension: GPT's analytical register.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

if "FIGS" not in dir():
    ROOT = Path("..").resolve()
    RESULTS = ROOT / "data" / "results"
    FIGS = RESULTS / "figures"
    FIGS.mkdir(parents=True, exist_ok=True)

NOISE_WORDS = {
    "monday", "tuesday", "wednesday", "thursday", "friday",
    "sapna", "reporter", "correspondent", "joined", "covering",
    "advertisement", "dealbook", "york",
    "china's", "tiktok's", "musk's", "ivory",
    "big", "soon", "going", "get", "day", "month", "last",
    "three", "percent", "house", "bill", "soccer", "students",
    "top", "little", "times", "wrote", "supported", "contributed",
}

lex_path = RESULTS / "results_lexical_fingerprint.csv"
lex = pd.read_csv(lex_path)
lex_clean = lex[~lex["word"].isin(NOISE_WORDS)].copy()
print(f"After filter: {len(lex_clean)} rows (was {len(lex)})")

# Only process overused — that's where the institutional voice signal lives
overused_pivot = lex_clean[lex_clean["type"] == "overused"].pivot_table(
    index="word", columns="prompt", values="log_ratio", aggfunc="mean"
).fillna(0)

# Make sure all 4 prompt columns exist
for col in ["A", "B", "D", "E"]:
    if col not in overused_pivot.columns:
        overused_pivot[col] = 0
overused_pivot = overused_pivot[["A", "B", "D", "E"]]

# Top 15 over
overused_pivot["mean"] = overused_pivot[["A", "B", "D", "E"]].mean(axis=1)
overused_top = overused_pivot.nlargest(15, "mean").drop(columns="mean")

print("\nTop 15 OVERUSED (after filter):")
print(overused_top.round(2))

# Plot
fig, ax = plt.subplots(figsize=(5.5, 6))

vmax = overused_top.values.max()
im = ax.imshow(
    overused_top.values, cmap="Reds",
    aspect="auto", vmin=0, vmax=vmax,
)

ax.set_xticks(range(len(overused_top.columns)))
ax.set_xticklabels([f"V.{c}" for c in overused_top.columns])
ax.set_yticks(range(len(overused_top.index)))
ax.set_yticklabels(overused_top.index, fontsize=10)

for i in range(len(overused_top.index)):
    for j in range(len(overused_top.columns)):
        val = overused_top.values[i, j]
        color = "white" if val > vmax * 0.5 else "black"
        ax.text(j, i, f"{val:+.1f}", ha="center", va="center",
                color=color, fontsize=8)

ax.set_title(
    "Lexical signature of GPT-4o-mini's analytical register\n"
    "(top 15 words overused vs NYT, by prompt)",
    fontsize=10, pad=12,
)

cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
cbar.set_label("log(GPT freq / NYT freq)", fontsize=9)

plt.tight_layout()
fig.savefig(FIGS / "fig3_lexical_fingerprint.pdf", bbox_inches="tight")
fig.savefig(FIGS / "fig3_lexical_fingerprint.png", bbox_inches="tight", dpi=200)
plt.show()

print(f"\nSaved cleaned Figure 3 to {FIGS / 'fig3_lexical_fingerprint.pdf'}")

In [ ]:

# =============================================================================
# CELL 4: Figure 4 - Reported-detail density (quantified interpretive flattening)
# Counts concrete reported detail (numbers, named-entity proxy, direct quotes)
# + sentence length across ALL generations vs NYT. Turns the n=20 manual audit
# into a scalable, cross-model number.
# =============================================================================

import re
from collections import Counter

_QUOTE = re.compile(r'[“”"]')          # direct-quote marks
_NUM = re.compile(r'\d')                          # digits (stats / dates / figures)
_CAP = re.compile(r"\b[A-Z][a-zA-Z'\.]+")         # capitalized tokens (named-entity proxy)
_SENT = re.compile(r'[.!?]+')
_WORD = re.compile(r"[A-Za-z']+")
GEN = ROOT / "data" / "generations"

def detail_metrics(text):
    if not isinstance(text, str) or not text.strip():
        return None
    words = _WORD.findall(text); nw = len(words)
    if nw < 50:
        return None
    sents = [s for s in _SENT.split(text) if s.strip()]
    caps_mid = max(len(_CAP.findall(text)) - len(sents), 0)  # minus ~1 sentence-initial cap per sentence
    return dict(numbers=1000 * len(_NUM.findall(text)) / nw,
                entities=1000 * caps_mid / nw,
                quotes=1000 * len(_QUOTE.findall(text)) / nw,
                sent_len=nw / max(len(sents), 1))

def corpus_means(df, col):
    rows = [detail_metrics(t) for t in df[col]]
    return pd.DataFrame([r for r in rows if r]).mean()

nyt_src = pd.read_csv(PROCESSED / "experiment_sample.csv")
detail = {"NYT (real)": corpus_means(nyt_src, "full_text")}
for label, fn in [(M_GPT, "generations_gpt.csv"), (M_G25, "generations_gemini.csv"),
                  (M_CLAUDE, "generations_claude.csv")]:
    p = GEN / fn
    if p.exists():
        g = pd.read_csv(p)
        g = g[~g["generated_full_text"].astype(str).str.startswith("[ERROR")]
        detail[label] = corpus_means(g, "generated_full_text")

detail_df = pd.DataFrame(detail).T
pct = detail_df / detail_df.loc["NYT (real)"] * 100
pct.to_csv(RESULTS / "results_reported_detail.csv")

metrics_order = ["numbers", "entities", "quotes", "sent_len"]
metric_labels = ["Numbers\n(stats/dates)", "Named\nentities", "Direct\nquotes", "Sentence\nlength"]
models_p = [m for m in [M_GPT, M_G25, M_CLAUDE] if m in pct.index]

fig, ax = plt.subplots(figsize=(8, 4.6))
x = np.arange(len(metrics_order)); w = 0.8 / len(models_p)
for i, m in enumerate(models_p):
    vals = [pct.loc[m, k] for k in metrics_order]
    bars = ax.bar(x + (i - (len(models_p) - 1) / 2) * w, vals, w, label=m,
                  color=color_map[m], alpha=0.9)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 1.5, f"{v:.0f}", ha="center",
                va="bottom", fontsize=7.5, color=color_map[m])
ax.axhline(100, color="gray", ls="--", lw=1.2)
ax.text(len(metrics_order) - 0.45, 101, "NYT = 100", color="gray", fontsize=9,
        ha="right", va="bottom", fontstyle="italic")
ax.set_xticks(x); ax.set_xticklabels(metric_labels)
ax.set_ylabel("Density relative to NYT (%)")
ax.set_title("Generated news carries far less reported detail than NYT\n"
             "(fewer numbers, named entities, quotes — packed into longer sentences; n = 40/desk)",
             fontsize=10, pad=10)
ax.legend(frameon=False, fontsize=9, loc="upper center", ncol=3)
ax.set_ylim(0, 145)
plt.tight_layout()
fig.savefig(FIGS / "fig4_reported_detail.pdf", bbox_inches="tight")
fig.savefig(FIGS / "fig4_reported_detail.png", bbox_inches="tight", dpi=200)
plt.show()
print(f"Saved Figure 4 to {FIGS}")

In [ ]:

# =============================================================================
# CELL 5: Figure 5 - the shared "LLM analytical register" (the robust tell)
# Words that GPT, Gemini AND Claude all over-use vs NYT. Per-model "signatures"
# proved noisy/topic-contaminated; the shared register is clean and robust, and
# ties directly to the flattening result (generic attribution vs named sources).
# =============================================================================

from collections import Counter
from matplotlib.patches import Patch

_W = re.compile(r"[A-Za-z][A-Za-z\-]+")   # no apostrophe -> drops possessives/contractions
_STOP = set(("the a an and or but is are was were be been being have has had do does did will would could should "
 "may might must can of to in on at by for with from as that this these those it its their they them we our us you "
 "your i my me he his him she her not no if than then so because while when where what which who whom how all any "
 "some more most other such only own same very just also still yet even now after before during between through over "
 "under above below out up down off into onto about said say says one two new year years first would there").split())

def _tk(t):
    return [w.lower() for w in _W.findall(str(t)) if w.lower() not in _STOP and len(w) > 3]

def _cnt(df, col):
    c = Counter()
    for t in df[col]:
        c.update(_tk(t))
    return c

nyt_src = pd.read_csv(PROCESSED / "experiment_sample.csv")
nyt_c = _cnt(nyt_src, "full_text")

mc = {}
for m, fn in [(M_GPT, "generations_gpt.csv"), (M_G25, "generations_gemini.csv"),
              (M_CLAUDE, "generations_claude.csv")]:
    p = GEN / fn
    if p.exists():
        g = pd.read_csv(p)
        g = g[~g["generated_full_text"].astype(str).str.startswith("[ERROR")]
        mc[m] = _cnt(g, "generated_full_text")
present = list(mc)

def _rate(c, w):
    return (c.get(w, 0) + 1) / sum(c.values())

allw = set(nyt_c)
for m in present:
    allw |= set(mc[m])
rows = []
for w in allw:
    if sum(mc[m].get(w, 0) for m in present) < 60:
        continue
    folds = [_rate(mc[m], w) / _rate(nyt_c, w) for m in present]
    if min(folds) > 1.0:            # over-used in EVERY model
        rows.append((w, float(np.mean(folds))))
rows = sorted(rows, key=lambda z: -z[1])[:12]
words = [w for w, _ in rows][::-1]
folds = [f for _, f in rows][::-1]
pd.DataFrame(rows, columns=["word", "mean_fold_vs_nyt"]).to_csv(
    RESULTS / "results_shared_register.csv", index=False)

ATTR = {"observers", "analysts", "spokesperson", "officials", "experts", "authorities", "sources"}
C_REG, C_ATTR = "#4C72B0", "#DD8452"
colors = [C_ATTR if w in ATTR else C_REG for w in words]

fig, ax = plt.subplots(figsize=(7.6, 4.8))
y = np.arange(len(words))
ax.barh(y, folds, color=colors, alpha=0.9)
for yi, f in zip(y, folds):
    ax.text(f + max(folds) * 0.01, yi, f"{f:.1f}×", va="center", fontsize=8.5, color="#333")
ax.axvline(1, color="gray", ls="--", lw=1)
ax.set_yticks(y); ax.set_yticklabels(words, fontsize=11)
ax.set_xlabel("Mean frequency vs NYT  (× = how many times more often than real NYT)")
ax.set_title("All three models share an abstract analytical register rare in NYT\n"
             "(words over-used by GPT, Gemini AND Claude; generic attribution highlighted)",
             fontsize=10, pad=10)
ax.legend(handles=[Patch(color=C_REG, label="analytical register"),
                   Patch(color=C_ATTR, label="generic attribution (vs NYT's named sources)")],
          frameon=False, fontsize=9, loc="lower right")
ax.set_xlim(0, max(folds) * 1.2)
plt.tight_layout()
fig.savefig(FIGS / "fig5_shared_register.pdf", bbox_inches="tight")
fig.savefig(FIGS / "fig5_shared_register.png", bbox_inches="tight", dpi=200)
plt.show()
print(f"Saved Figure 5 (shared register) to {FIGS}")

In [ ]:
# =============================================================================
# CELL 5: Summary
# =============================================================================

print("\n" + "=" * 60)
print("FIGURES GENERATED")
print("=" * 60)
for f in sorted(FIGS.glob("*")):
    size = f.stat().st_size
    print(f"  {f.name:40s}  {size:>8,} bytes")
print()
print("Open the .pdf versions in your paper LaTeX. PNG previews available.")